# 1. Modeling Team, Meta Shift and Games

 League of Legend is a game with the following specificity:

- Team based confrontation: 5 versus 5
- Unbalanced: blue versus red side
- Role with different impact on the game: Toplaner, Jungler, Midlaner, ADC, Support
- Meta: frequent patches alter players performances.

To bets emmulate a league match, we need to work on the core of the simulation: Player, Match, Solver.
This 3 notions interact closely to produce synthetic game results.


## Team
The base component to represent competitor is the SPlayer, an interface. Anything that provides a name() and a level() can:

- play match
- participates in tournament
- be ranked.

This is what we will use to represent teams. You can build your own simple class from scratch, or you can inherit from rstt base component. here, because we have meta shift that will affect the level of teams, a good choice is to inherit from the abstract class PlayerTVS

#### TODO:

Implement the LoLTeam class in model/lolteam.py




https://www.leagueoflegends.com/en-us/how-to-play/
https://lolesports.com/en-US/news/dev-diary-unveiling-the-global-power-rankings
https://liquipedia.net/leagueoflegends/Main_Page

In [1]:
from rstt import BasicPlayer, GaussianPlayer

from project.model import LoLTeam
from project.scene import Role

nameA = "TeamA"
playersA = {role: GaussianPlayer(name=f"Player_{role}") for role in Role}
teamA = LoLTeam(name=nameA, players=playersA)

nameB = "TeamB"
playersB = {role: BasicPlayer(name=f"Player_{role}") for role in Role}
teamB = LoLTeam(name=nameB, players=playersB)

In [2]:
from rstt import Duel, BetterWin

# Base fonctionality
assert teamA.name() == nameA
assert teamB.name() == nameB

# Must be able to update level
meta_1_level_A = teamA.level()
teamA.update_level()
meta_2_level_A = teamA.level()
assert meta_1_level_A != meta_2_level_A

# Must be able to have constant level - without crashing
meta_1_level_B = teamB.level()
teamB.update_level()
meta_2_level_B = teamB.level()
assert meta_1_level_B == meta_2_level_B

# Can play games
duel = Duel(teamA, teamB)
BetterWin().solve(duel)
assert not duel.live()

In [3]:
from rstt import BTRanking, SingleEliminationBracket

other_teams = BasicPlayer.create(nb=14)
all_teams = [teamA, teamB] + other_teams

# check ranking can register lolteam with proper rating
groundtruth = BTRanking(name="GroundTruth", players=all_teams)
gt_level_A = groundtruth.point(teamA)
assert gt_level_A == teamA.level()

# check ranking keeps track of lolteam level update
teamA.update_level()
groundtruth.update(player=teamA)
gt_new_level_A = groundtruth.point(teamA)
assert gt_new_level_A != gt_level_A

# check lolteam can play a tournament
cup = SingleEliminationBracket("test cup", groundtruth, BetterWin())
cup.registration(all_teams)
cup.run()

## Meta Implementation

We want to seperate the skills of team that can fluctuate due to internal mechanism, such as:
- improvement / regression -> LogisticPlayer, ExponentialPlayer
- irregular performances -> GaussianPlayer, CyclePlayer

from external mechanism (side and meta effects). This enable rich teams behaviour, as one can update teams level and meta shifts at different frequencies (per Game/Tournament/Season/Year). And fine tune model parameters accoringly. 

One trick about rstt type system is that the dynamic typechecker verify only the existence of methods for Protocol, not their function signatures, which allows flexible design. In this case we can alter team's skills by passing weigths to the level function without breaking the entire simulation engine. Just make sure there is a default value for any parameters you add.

#### TODO:
- implement a MetaData class that can affect the the level of a team based on the impact each role has.

In [4]:
import numpy as np
from project.model import MetaData

# different level with and without meta impact
meta = MetaData(weights={role: weight for role, weight in zip(Role, np.random.uniform(0,1,5))})
pre_shift_level = teamA.level(meta.weights())
team_level = teamA.level()
assert pre_shift_level != team_level

# different level before and after meta update
meta.update()
post_shift_level = teamA.level(meta.weights())
assert pre_shift_level != post_shift_level

# meta update did not alter team intrasect level
assert team_level == teamA.level()

## Solver Implementation (Games)

The solver is responsible to assign a score to a game. The simple approach to implement a custom Solver is to inherit from the ScoreProb class and define:
- The possible outcomes. A game outcome [Score]() is a list of float value that represent each involved players, their game result. For Duel there are Built-in Score available. WIN is just an alias for [1.0, 0.0].
- A probabilitie function that takes has input a SMatch instance and return a list of probability for each possible outcomes.

LoL games are decisifs, no draws. Teams either win or lose. The game results probabilities must be level based and take into account the meta state. Because we are building a simulation interacting with an elo like ranking, a suited probability function could be an option , but this up to you!

#### TODO:
- Create a Solver that assign game score to LoLTeam.
- Meta should influence the level of a team based on individual players level and role within the team
- Red/Blue side should modify wins probabilities.

In [5]:
from rstt import WIN, LOSE
from project.model import LoLSolver

# Instanciate your solver
meta = MetaData()
solver = LoLSolver(meta=meta)
lol_game = Duel(teamA, teamB)

# check that the solver plays a game
solver.solve(lol_game)
assert not lol_game.live()

# validate game outcomes
assert lol_game.scores() in [WIN, LOSE]

# check that the solver can not play an already played game
try:
    solver.solve(lol_game)
    assert False, "Your solver affect already played games, your code breaks the framework rules"
except RuntimeError:
    pass

## Model verification

Tests are important in Simulation Design. One must ensure that the behaviours of the models are well defined.
We are testing extreme cases of our system. If roles impact outcomes then a single, strong enough player, can sine decide the outcome of games.

#### TODO:

- make all "100% win rate" test succeed.

In [6]:
# helper to create unballanced meta
def broken_role(_1v9: Role):
    weights = {role: 0 for role in Role}
    weights.update({_1v9: 1})
    return weights

# Team/Meta Settings
blue = BasicPlayer("BLUE", level=1500)
red = BasicPlayer("RED", level=1500)
meta = MetaData(blue=blue, red=red)
solver = LoLSolver(meta)


# Test settings
test_samples = 1000
error_rate = 0.01

# ------------------- #
# 1. TEST ROLE IMPACT #
# ------------------- #
players_team1 = {role: teamA.player(role) for role in Role }
players_team2 = {role: teamB.player(role) for role in Role }
smurf = BasicPlayer(name=f"SMURFING", level=3000)
inter = BasicPlayer(name=f"BOOSTED", level=0)

for role in Role:
    meta.set_weights(broken_role(role))
    
    players_team1.update({role: smurf})
    players_team2.update({role: inter})
    carried_team = LoLTeam(name="lucky", players=players_team1)
    inted_team = LoLTeam(name="unlucky", players=players_team2)

    fails = 0
    for i in range(test_samples):
        # test one side
        not_fun_game = Duel(carried_team, inted_team)
        solver.solve(not_fun_game)
        msg = f"{role}, did not carry the {i}-the game"
        
        if not_fun_game.winner() != carried_team:
            fails += 1

        # test the other side
        not_fun_game = Duel(inted_team, carried_team)
        solver.solve(not_fun_game)
        msg = f"{role}, did not carry the {i}-the game"
        if not_fun_game.winner() != carried_team:
            fails += 1

    if fails / (2*test_samples) <= error_rate:
        print(f"✅ {role} did carry games.")
    else:
        print(f"❌ {role} did not carry games")

✅ Toplaner did carry games.
✅ Jungler did carry games.
✅ Midlaner did carry games.
✅ Botlaner did carry games.
✅ Support did carry games.


In [7]:
# ------------------------------------ #
# 2. TEST Blue side is unfairly strong #
# ------------------------------------ #
blue = BasicPlayer("BLUE", level=3000)
red = BasicPlayer("RED", level=0)
meta = MetaData(blue=blue, red=red)
solver = LoLSolver(meta)

for i in range(test_samples):
    A_wins = Duel(teamA, teamB)
    solver.solve(A_wins)
    if A_wins.winner() != teamA:
        fails += 1
   
    B_wins = Duel(teamB, teamA)
    solver.solve(B_wins)
    if B_wins.winner() != teamB:
        fails += 1

if fails / (2*test_samples) <= error_rate:
    print("✅ Blue side wins games.")
else:
    print("❌ Blue side does not win games")

✅ Blue side wins games.


In [8]:
# ----------------------------------- #
# 2. TEST Red side is unfairly strong #
# ----------------------------------- #
blue = BasicPlayer("BLUE", level=0)
red = BasicPlayer("RED", level=3000)
meta = MetaData(blue=blue, red=red)
solver = LoLSolver(meta)

for i in range(test_samples):
    A_wins = Duel(teamB, teamA)
    solver.solve(A_wins)
    if A_wins.winner() != teamA:
        fails += 1
        
    B_wins = Duel(teamA, teamB)
    solver.solve(B_wins)
    if B_wins.winner() != teamB:
        fails += 1

if fails / (2*test_samples) <= error_rate:
    print("✅ Red side wins games.")
else:
    print("❌ Red side does not win games")

✅ Red side wins games.
